In [4]:
install.packages("hoopR")


The downloaded binary packages are in
	/var/folders/sz/k_920m494wd37gp8s3kxnlxw0000gn/T//Rtmp76WKqd/downloaded_packages


In [5]:
library(hoopR)
library(dplyr)
library(readr)
library(tidyr)

teams_raw <- espn_nba_teams()

flat <- teams_raw %>%
  tidyr::unnest_wider(team, names_sep = "_", names_repair = "unique")

# Helper: pick the first column that exists from a list of candidates
pick_col <- function(df, candidates) {
  nm <- intersect(candidates, names(df))
  if (length(nm) == 0) return(rep(NA_character_, nrow(df)))
  as.character(df[[nm[1]]])
}

teams <- tibble::tibble(
  team_id   = pick_col(flat, c("team_id", "id", "team__id", "team_id_1")),
  team_uid  = pick_col(flat, c("team_uid", "uid", "team__uid", "team_uid_1")),
  team_code = pick_col(flat, c("team_abbreviation", "abbreviation", "team__abbreviation")),
  team_name = pick_col(flat, c("team_displayName", "displayName", "team__displayName",
                               "team_name", "name", "shortDisplayName", "team_shortDisplayName")),
  team_slug = pick_col(flat, c("team_slug", "slug", "team__slug"))
) %>%
  distinct(team_id, team_code, team_name, .keep_all = TRUE)

write_csv(teams, "nba_teams_hoopR.csv")
print(teams)

# Optional: see exactly what columns you got after unnesting
# print(names(flat))

# A tibble: 30 × 5
   team_id team_uid team_code team_name team_slug
   <chr>   <chr>    <chr>     <chr>     <chr>    
 1 1       NA       ATL       NA        NA       
 2 2       NA       BOS       NA        NA       
 3 17      NA       BKN       NA        NA       
 4 30      NA       CHA       NA        NA       
 5 4       NA       CHI       NA        NA       
 6 5       NA       CLE       NA        NA       
 7 6       NA       DAL       NA        NA       
 8 7       NA       DEN       NA        NA       
 9 8       NA       DET       NA        NA       
10 9       NA       GS        NA        NA       
# ℹ 20 more rows


In [9]:
#!/usr/bin/env Rscript

suppressPackageStartupMessages({
  library(hoopR)
  library(dplyr)
  library(readr)
  library(stringr)
  library(purrr)
})

SEASONS <- 2026:2026  # season END years (2009-10 .. 2017-18)
BASE_OUT_DIR <- file.path("..", "data", "games_live2")
dir.create(BASE_OUT_DIR, recursive = TRUE, showWarnings = FALSE)

sanitize_team <- function(x) {
  x <- as.character(x)
  x <- toupper(str_trim(x))
  x <- gsub("[^A-Z0-9]+", "_", x)
  x <- gsub("_+", "_", x)
  x <- gsub("^_|_$", "", x)
  ifelse(is.na(x) | x == "", "UNK", x)
}

load_schedule_safely <- function(season_end_year) {
  candidates <- list(
    function() hoopR::load_nba_schedule(seasons = season_end_year),
    function() hoopR::nba_schedule(seasons = season_end_year)   # some versions differ
  )
  for (fn in candidates) {
    out <- try(fn(), silent = TRUE)
    if (!inherits(out, "try-error") && is.data.frame(out) && nrow(out) > 0) return(out)
  }
  stop(sprintf("Could not load NBA schedule metadata via hoopR for season ending %d.", season_end_year))
}

pick_col <- function(df, candidates) {
  cand <- candidates[candidates %in% names(df)]
  if (length(cand) == 0) return(NA_character_)
  cand[[1]]
}

process_one_season <- function(season_end_year) {
  season_dir <- file.path(BASE_OUT_DIR, as.character(season_end_year))
  dir.create(season_dir, recursive = TRUE, showWarnings = FALSE)

  message(sprintf("\n==============================\nSeason ending %d\n==============================", season_end_year))
  message(sprintf("Loading PBP for season ending %d ...", season_end_year))

  pbp <- hoopR::load_nba_pbp(seasons = season_end_year) %>%
    mutate(game_id = as.character(game_id))

  if (!"game_id" %in% names(pbp)) stop("PBP is missing `game_id`.")

  game_ids <- unique(pbp$game_id)
  message(sprintf("PBP rows: %d | games: %d", nrow(pbp), length(game_ids)))

  message("Loading schedule metadata to map game_id -> teams ...")
  sched <- load_schedule_safely(season_end_year)

  nm <- names(sched)

  gid_col <- c("game_id", "gameId", "id", "espn_game_id", "gameID")[
    c("game_id", "gameId", "id", "espn_game_id", "gameID") %in% nm
  ][1]
  if (is.na(gid_col)) stop("Schedule metadata did not contain a recognizable game id column.")

  sched <- sched %>% mutate(game_id = as.character(.data[[gid_col]]))

  away_col <- pick_col(sched, c("away_team_abbreviation","away_abbreviation","away_team","awayTeamAbbreviation","away_team_abb","away_abbr"))
  home_col <- pick_col(sched, c("home_team_abbreviation","home_abbreviation","home_team","homeTeamAbbreviation","home_team_abb","home_abbr"))

  away_name_col <- pick_col(sched, c("away_team_name","awayTeamName","away_name","awayTeam","away_team_full"))
  home_name_col <- pick_col(sched, c("home_team_name","homeTeamName","home_name","homeTeam","home_team_full"))

  date_col <- pick_col(sched, c("game_date","date","start_date","gameDate","start_time","startTimeUTC"))

  games_meta <- sched %>%
      transmute(
        game_id,
        away = if (!is.na(away_col)) .data[[away_col]] else if (!is.na(away_name_col)) .data[[away_name_col]] else NA_character_,
        home = if (!is.na(home_col)) .data[[home_col]] else if (!is.na(home_name_col)) .data[[home_name_col]] else NA_character_
      )
    
    games_meta$game_date <- if (!is.na(date_col)) as.Date(sched[[date_col]]) else as.Date(NA)
    
    games_meta <- games_meta %>%
      mutate(
        away = sanitize_team(away),
        home = sanitize_team(home)
      ) %>%
      distinct(game_id, .keep_all = TRUE)

  latest_game_date <- games_meta %>%
      filter(game_id %in% game_ids) %>%
      filter(!is.na(game_date)) %>%
      summarise(max_date = max(game_date)) %>%
      pull(max_date)
    
    if (length(latest_game_date) > 0 && !is.na(latest_game_date)) {
      message(sprintf("Most recent game in dataset: %s", latest_game_date))
    } else {
      message("Most recent game in dataset: UNKNOWN (no usable game date column found)")
    }
    
    pbp2 <- pbp %>% left_join(games_meta, by = "game_id")

  missing_team_ids <- pbp2 %>%
    distinct(game_id, away, home) %>%
    filter(away == "UNK" | home == "UNK")

  if (nrow(missing_team_ids) > 0) {
    message(sprintf("Warning: %d games missing team info (UNK).", nrow(missing_team_ids)))
  }

  message(sprintf("Writing per-game CSVs to %s ...", season_dir))

  written <- 0L
  pbp2 %>%
    group_by(game_id, away, home) %>%
    group_walk(function(df, key) {
      gid  <- as.character(key$game_id[[1]])
      away <- as.character(key$away[[1]])
      home <- as.character(key$home[[1]])

      filename <- sprintf("%s_%s_%s.csv", gid, away, home)
      path <- file.path(season_dir, filename)

      write_csv(df, path, na = "")
      written <<- written + 1L
      if (written %% 50L == 0L) message(sprintf("...written %d", written))
    })

  message(sprintf("Done season %d. Wrote %d files.", season_end_year, written))
  invisible(written)
}

total_written <- 0L

for (season_end_year in SEASONS) {
  wrote <- tryCatch(
    process_one_season(season_end_year),
    error = function(e) {
      message(sprintf("ERROR in season %d: %s", season_end_year, conditionMessage(e)))
      0L
    }
  )
  total_written <- total_written + wrote
}

message(sprintf("\nAll done. Total files written across seasons: %d", total_written))


Season ending 2026

Loading PBP for season ending 2026 ...

PBP rows: 460089 | games: 946

Loading schedule metadata to map game_id -> teams ...

Most recent game in dataset: 2026-03-06

Writing per-game CSVs to ../data/games_live2/2026 ...

...written 50

...written 100

...written 150

...written 200

...written 250

...written 300

...written 350

...written 400

...written 450

...written 500

...written 550

...written 600

...written 650

...written 700

...written 750

...written 800

...written 850

...written 900

Done season 2026. Wrote 946 files.


All done. Total files written across seasons: 946



In [8]:
seasons <- 2026:2026

for (season in seasons) {
  cat("Fetching season", season, "...\n")
  
  tryCatch({
    schedule <- load_nba_schedule(seasons = season)
    
    # Build filename: 2003 -> "schedule_2002-03.csv", 2010 -> "schedule_2009-10.csv"
    start_year <- season - 1
    end_yy <- sprintf("%02d", season %% 100)
    filename <- paste0("schedule_", start_year, "-", end_yy, ".csv")
    
    write.csv(schedule, file.path("../data/schedules2", filename), row.names = FALSE)
    cat("  Saved:", filename, "-", nrow(schedule), "rows\n")
  }, error = function(e) {
    cat("  ERROR for season", season, ":", conditionMessage(e), "\n")
  })
  
  Sys.sleep(1)  # be polite to the API
}

cat("Done!\n")

Fetching season 2026 ...


Warning message in file(file, ifelse(append, "a", "w")):
“cannot open file '../data/schedules2/schedule_2025-26.csv': No such file or directory”


  ERROR for season 2026 : cannot open the connection 
Done!


In [8]:
# =========================
# Team stats builder (hoopR)
# =========================

suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
  library(stringr)
  library(tidyr)
  library(purrr)
  library(janitor)
  library(rlang)
  library(hoopR)
})

SEASONS <- 2026:2026                       # season end-years (2021 == 2020-21)
SCHEDULE_DIR <- normalizePath("../data/schedules", winslash = "/", mustWork = TRUE)
OUT_PATH <- "data/team_stats.csv"
PRINT_SAMPLE_ROW <- TRUE                   # prints one sample row per season (turnovers check)

dir.create("data", showWarnings = FALSE, recursive = TRUE)

season_label <- function(season_end) sprintf("%d-%02d", season_end - 1L, season_end %% 100L)

schedule_path <- function(season_end) {
  file.path(SCHEDULE_DIR, paste0("schedule_", season_label(season_end), ".csv"))
}

# ---- Read schedule -> only game_ids (we use hoopR's team_home_away for home/away) ----
read_schedule_game_ids <- function(path) {
  if (!file.exists(path)) stop("Missing schedule file: ", path)

  first_line <- read_lines(path, n_max = 1)
  has_header <- !str_detect(first_line, "^\\s*\\d")  # header if first char isn't a digit

  sch <- read_csv(path, col_names = has_header, show_col_types = FALSE) |> clean_names()

  gid <- if ("game_id" %in% names(sch)) sch$game_id else sch[[1]]
  tibble(game_id = as.character(gid)) |>
    filter(!is.na(game_id), game_id != "") |>
    distinct(game_id)
}

# ---- hoopR loader (bulk per season) ----
fetch_team_box_for_season <- function(season_end) {
  if (exists("load_nba_team_box", where = asNamespace("hoopR"), inherits = FALSE)) {
    hoopR::load_nba_team_box(seasons = season_end)
  } else if (exists("load_nba_team_boxscores", where = asNamespace("hoopR"), inherits = FALSE)) {
    hoopR::load_nba_team_boxscores(seasons = season_end)
  } else {
    stop(
      "Couldn't find a team boxscore loader in hoopR.\n",
      "Run: ls('package:hoopR')\n",
      "…and swap in the correct function name above."
    )
  }
}

# ---- append helper ----
append_csv <- function(df, path) {
  if (!file.exists(path)) {
    write_csv(df, path)
  } else {
    write.table(df, file = path, sep = ",", row.names = FALSE, col.names = FALSE, append = TRUE)
  }
}

# =========================
# Main
# =========================

# If output exists, skip already-done games
already_done <- character(0)
if (file.exists(OUT_PATH)) {
  existing <- read_csv(OUT_PATH, show_col_types = FALSE) |> clean_names()
  if ("game_id" %in% names(existing)) already_done <- unique(as.character(existing$game_id))
  message("Found existing output with ", length(already_done), " games; will skip those.")
}

# Final output column order (NO MIN, NO PLUS_MINUS)
col_order <- c(
  "game_id","game_date","season",
  "home_team","away_team",
  "home_team_full","away_team_full",
  "away_TEAM_CITY",
  "away_FGM","away_FGA","away_FG_PCT","away_FG3M","away_FG3A","away_FG3_PCT",
  "away_FTM","away_FTA","away_FT_PCT","away_OREB","away_DREB","away_REB","away_AST","away_STL",
  "away_BLK","away_TO","away_PF","away_PTS",
  "home_TEAM_CITY",
  "home_FGM","home_FGA","home_FG_PCT","home_FG3M","home_FG3A","home_FG3_PCT",
  "home_FTM","home_FTA","home_FT_PCT","home_OREB","home_DREB","home_REB","home_AST","home_STL",
  "home_BLK","home_TO","home_PF","home_PTS"
)

for (season_end in SEASONS) {
  lbl <- season_label(season_end)
  p <- schedule_path(season_end)
  message("\nReading schedule: ", p)

  schedule_ids <- read_schedule_game_ids(p)
  if (nrow(schedule_ids) == 0) {
    message("No game_ids found in schedule for ", lbl, "; skipping.")
    next
  }

  # remove games already in output
  schedule_ids <- schedule_ids |> filter(!game_id %in% already_done)
  if (nrow(schedule_ids) == 0) {
    message("All schedule games already present for ", lbl, "; skipping.")
    next
  }

  message("Fetching hoopR team box for season ", lbl, " (end-year ", season_end, ") ...")
  tb_raw <- fetch_team_box_for_season(season_end)

  if (PRINT_SAMPLE_ROW) {
    one <- tb_raw %>% dplyr::slice(1)
    cat("\n--- Sample row turnovers check (", lbl, ") ---\n", sep = "")
    print(one %>% dplyr::select(
      game_id, game_date, team_abbreviation, team_home_away,
      team_score, opponent_team_score,
      turnovers, team_turnovers, total_turnovers
    ))
  }

  tb <- tb_raw |>
    janitor::clean_names() |>
    mutate(game_id = as.character(game_id)) |>
    semi_join(schedule_ids, by = "game_id") |>
    mutate(
      season = lbl,
      game_date = as.Date(game_date)
    ) |>
    transmute(
      game_id,
      game_date,
      season,
      side      = team_home_away,         # "home" / "away"
      team      = team_abbreviation,      # PHX
      team_full = team_display_name,      # Phoenix Suns
      team_city = team_location,          # Phoenix

      # Stats you want (canonical turnovers = `turnovers`)
      fgm    = as.numeric(field_goals_made),
      fga    = as.numeric(field_goals_attempted),
      fg_pct = as.numeric(field_goal_pct),
      fg3m   = as.numeric(three_point_field_goals_made),
      fg3a   = as.numeric(three_point_field_goals_attempted),
      fg3_pct= as.numeric(three_point_field_goal_pct),
      ftm    = as.numeric(free_throws_made),
      fta    = as.numeric(free_throws_attempted),
      ft_pct = as.numeric(free_throw_pct),
      oreb   = as.numeric(offensive_rebounds),
      dreb   = as.numeric(defensive_rebounds),
      reb    = as.numeric(total_rebounds),
      ast    = as.numeric(assists),
      stl    = as.numeric(steals),
      blk    = as.numeric(blocks),
      to     = as.numeric(turnovers),
      pf     = as.numeric(fouls),
      pts    = as.numeric(team_score)
    ) |>
    # ESPN pct columns often come as percent (e.g., 44.2). Convert to fraction.
    mutate(
      fg_pct  = ifelse(!is.na(fg_pct)  & fg_pct  > 1, fg_pct  / 100, fg_pct),
      fg3_pct = ifelse(!is.na(fg3_pct) & fg3_pct > 1, fg3_pct / 100, fg3_pct),
      ft_pct  = ifelse(!is.na(ft_pct)  & ft_pct  > 1, ft_pct  / 100, ft_pct)
    ) |>
    filter(side %in% c("home", "away"))

  if (nrow(tb) == 0) {
    message("No matching games after filtering for ", lbl, "; skipping.")
    next
  }

  wide <- tb |>
    pivot_wider(
      id_cols = c(game_id, game_date, season),
      names_from = side,
      values_from = c(team, team_full, team_city,
                      fgm, fga, fg_pct, fg3m, fg3a, fg3_pct,
                      ftm, fta, ft_pct, oreb, dreb, reb, ast, stl, blk, to, pf, pts),
      names_glue = "{side}_{.value}"
    )

  out <- wide |>
    transmute(
      game_id, game_date, season,
      home_team      = home_team,
      away_team      = away_team,
      home_team_full = home_team_full,
      away_team_full = away_team_full,

      away_TEAM_CITY = away_team_city,
      away_FGM = away_fgm,
      away_FGA = away_fga,
      away_FG_PCT = away_fg_pct,
      away_FG3M = away_fg3m,
      away_FG3A = away_fg3a,
      away_FG3_PCT = away_fg3_pct,
      away_FTM = away_ftm,
      away_FTA = away_fta,
      away_FT_PCT = away_ft_pct,
      away_OREB = away_oreb,
      away_DREB = away_dreb,
      away_REB = away_reb,
      away_AST = away_ast,
      away_STL = away_stl,
      away_BLK = away_blk,
      away_TO  = away_to,
      away_PF  = away_pf,
      away_PTS = away_pts,

      home_TEAM_CITY = home_team_city,
      home_FGM = home_fgm,
      home_FGA = home_fga,
      home_FG_PCT = home_fg_pct,
      home_FG3M = home_fg3m,
      home_FG3A = home_fg3a,
      home_FG3_PCT = home_fg3_pct,
      home_FTM = home_ftm,
      home_FTA = home_fta,
      home_FT_PCT = home_ft_pct,
      home_OREB = home_oreb,
      home_DREB = home_dreb,
      home_REB = home_reb,
      home_AST = home_ast,
      home_STL = home_stl,
      home_BLK = home_blk,
      home_TO  = home_to,
      home_PF  = home_pf,
      home_PTS = home_pts
    ) |>
    distinct(game_id, .keep_all = TRUE)

  # Ensure exact schema (add any missing columns as NA, order them)
  for (nm in col_order) if (!nm %in% names(out)) out[[nm]] <- NA
  out <- out |> select(all_of(col_order))

  message("Appending ", nrow(out), " games to ", OUT_PATH)
  append_csv(out, OUT_PATH)

  already_done <- c(already_done, out$game_id)
}

message("\nDone. Output written to: ", OUT_PATH)

Found existing output with 31571 games; will skip those.


Reading schedule: /Users/danielyang/Desktop/Extra_Curriculars/kalshi-bot/nba/data/schedules/schedule_2025-26.csv

Fetching hoopR team box for season 2025-26 (end-year 2026) ...




--- Sample row turnovers check (2025-26) ---


── ESPN NBA Team Boxscores from hoopR data repository ────── hoopR 2.1.0 ──

ℹ Data updated: 2026-02-23 04:40:30 PST



# A tibble: 1 × 9
    game_id game_date  team_abbreviation team_home_away team_score
      <int> <date>     <chr>             <chr>               <int>
1 401810682 2026-02-22 ORL               away                  111
# ℹ 4 more variables: opponent_team_score <int>, turnovers <int>,
#   team_turnovers <int>, total_turnovers <int>


No matching games after filtering for 2025-26; skipping.


Done. Output written to: data/team_stats.csv



In [12]:
# ===============================
# Game rosters builder (hoopR)
# ===============================

suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
  library(stringr)
  library(tidyr)
  library(purrr)
  library(janitor)
  library(rlang)
  library(hoopR)
})

SEASONS <- 2002:2026
SCHEDULE_DIR <- normalizePath("../data/schedules", winslash = "/", mustWork = TRUE)
OUT_DIR <- "data/game_rosters"
PRINT_COLS_ONCE <- TRUE   # print columns returned by hoopR for the first season only

dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

season_label <- function(season_end) sprintf("%d-%02d", season_end - 1L, season_end %% 100L)

schedule_path <- function(season_end) {
  file.path(SCHEDULE_DIR, paste0("schedule_", season_label(season_end), ".csv"))
}

# --- schedule reader: extracts game_id, game_date, away_team, home_team (from "AAA @ BBB") ---
read_schedule_meta <- function(path, season_end) {
  if (!file.exists(path)) stop("Missing schedule file: ", path)

  first_line <- read_lines(path, n_max = 1)
  has_header <- !str_detect(first_line, "^\\s*\\d")

  sch <- read_csv(path, col_names = has_header, show_col_types = FALSE) |> clean_names()

  gid <- if ("game_id" %in% names(sch)) sch$game_id else sch[[1]]
  gid <- as.character(gid)

  gdate <- if ("game_date" %in% names(sch)) {
    sch$game_date
  } else if ("date" %in% names(sch)) {
    sch$date
  } else {
    date_col_idx <- which(vapply(
      sch,
      function(x) mean(str_detect(as.character(x), "^\\d{4}-\\d{2}-\\d{2}$"), na.rm = TRUE),
      numeric(1)
    ) > 0.5)
    if (length(date_col_idx) == 0) NA else sch[[date_col_idx[1]]]
  }

  matchup_idx <- which(vapply(
    sch,
    function(x) any(str_detect(as.character(x), "\\b[A-Z]{3} @ [A-Z]{3}\\b")),
    logical(1)
  ))
  matchup <- if (length(matchup_idx) == 0) NA_character_ else as.character(sch[[matchup_idx[length(matchup_idx)]]])

  away <- home <- rep(NA_character_, length(gid))
  if (!all(is.na(matchup))) {
    parts <- str_split(matchup, " @ ", simplify = TRUE)
    if (ncol(parts) >= 2) {
      away <- parts[, 1]
      home <- parts[, 2]
    }
  }

  tibble(
    game_id    = gid,
    game_date  = as.Date(gdate),
    season     = season_label(season_end),
    season_end = season_end,
    away_team  = away,
    home_team  = home
  ) |>
    filter(!is.na(game_id), game_id != "") |>
    distinct(game_id, .keep_all = TRUE)
}

# ---- hoopR loader for player box/roster (bulk per season) ----
get_hoopr_loader <- function(candidates) {
  for (fn in candidates) {
    if (exists(fn, where = asNamespace("hoopR"), inherits = FALSE)) {
      return(get(fn, envir = asNamespace("hoopR")))
    }
  }
  NULL
}

fetch_player_box_for_season <- function(season_end) {
  loader <- get_hoopr_loader(c(
    "load_nba_player_box",
    "load_nba_player_boxscores",
    "load_nba_player_box_score",
    "load_nba_boxscore_players",
    "load_nba_players_box"
  ))

  if (is.null(loader)) {
    stop(
      "Couldn't find a player boxscore/roster loader in hoopR.\n",
      "Run: ls('package:hoopR')\n",
      "and tell me what functions look relevant (player/box/roster), and I'll wire it up."
    )
  }

  out <- tryCatch(loader(seasons = season_end), error = function(e1) {
    tryCatch(loader(season = season_end), error = function(e2) {
      stop("Found loader but couldn't call it with seasons/season args:\n", e1$message, "\n", e2$message)
    })
  })
  out
}

# ---- small helper for filenames ----
safe_abbr <- function(x) {
  x <- ifelse(is.na(x) | x == "", "UNK", x)
  x <- toupper(x)
  str_replace_all(x, "[^A-Z0-9]+", "")
}

# now includes season_end folder: data/game_rosters/{season_end}/
out_file_for_game <- function(season_end, game_id, away_team, home_team) {
  season_dir <- file.path(OUT_DIR, as.character(season_end))
  dir.create(season_dir, showWarnings = FALSE, recursive = TRUE)
  file.path(season_dir, sprintf("%s_%s_%s.csv", game_id, safe_abbr(away_team), safe_abbr(home_team)))
}

# ===============================
# Main
# ===============================

games_meta <- map_dfr(SEASONS, function(season_end) {
  p <- schedule_path(season_end)
  message("Reading schedule: ", p)
  read_schedule_meta(p, season_end)
}) |> distinct(game_id, .keep_all = TRUE)

message("Total games in schedules: ", nrow(games_meta))

for (season_end in SEASONS) {
  lbl <- season_label(season_end)
  meta_s <- games_meta |> filter(season_end == !!season_end)
  if (nrow(meta_s) == 0) next

  message("\nFetching player box/roster for season ", lbl, " (end-year ", season_end, ") ...")
  pb_raw <- fetch_player_box_for_season(season_end) |> clean_names()

  if (PRINT_COLS_ONCE) {
    message("Columns returned by hoopR player box/roster loader (first season only):")
    print(sort(names(pb_raw)))
    PRINT_COLS_ONCE <- FALSE
  }

  pb_raw <- pb_raw |>
    mutate(game_id = as.character(game_id)) |>
    semi_join(meta_s |> select(game_id), by = "game_id")

  if (nrow(pb_raw) == 0) {
    message("No roster rows matched schedule IDs for ", lbl, ". Skipping season.")
    next
  }

  meta_s_ids <- meta_s |> select(game_id, away_team, home_team)

  # Ensure season folder exists up front
  dir.create(file.path(OUT_DIR, as.character(season_end)), showWarnings = FALSE, recursive = TRUE)

  for (i in seq_len(nrow(meta_s_ids))) {
    gid  <- meta_s_ids$game_id[i]
    away <- meta_s_ids$away_team[i]
    home <- meta_s_ids$home_team[i]

    out_path <- out_file_for_game(season_end, gid, away, home)
    if (file.exists(out_path)) next

    gdf <- pb_raw |> filter(game_id == gid)
    if (nrow(gdf) == 0) next

    gdf <- gdf |>
      mutate(
        schedule_season_end = season_end,
        schedule_away_team = safe_abbr(away),
        schedule_home_team = safe_abbr(home)
      )

    write_csv(gdf, out_path)
  }

  message("Wrote game roster files for season ", lbl, " to ", file.path(OUT_DIR, as.character(season_end)))
}

message("\nDone. Game roster files saved under: ", OUT_DIR)

Reading schedule: /Users/danielyang/Desktop/Extra_Curriculars/kalshi-bot/nba/data/schedules/schedule_2001-02.csv

Reading schedule: /Users/danielyang/Desktop/Extra_Curriculars/kalshi-bot/nba/data/schedules/schedule_2002-03.csv

Reading schedule: /Users/danielyang/Desktop/Extra_Curriculars/kalshi-bot/nba/data/schedules/schedule_2003-04.csv

Reading schedule: /Users/danielyang/Desktop/Extra_Curriculars/kalshi-bot/nba/data/schedules/schedule_2004-05.csv

Reading schedule: /Users/danielyang/Desktop/Extra_Curriculars/kalshi-bot/nba/data/schedules/schedule_2005-06.csv

Reading schedule: /Users/danielyang/Desktop/Extra_Curriculars/kalshi-bot/nba/data/schedules/schedule_2006-07.csv

Reading schedule: /Users/danielyang/Desktop/Extra_Curriculars/kalshi-bot/nba/data/schedules/schedule_2007-08.csv

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details, e.g.:
  dat <- vroom(...)
  problems(dat)”
Reading schedule: /Users/danielyang/Desktop/Extra_Curriculars/k

 [1] "active"                            "assists"                          
 [3] "athlete_display_name"              "athlete_headshot_href"            
 [5] "athlete_id"                        "athlete_jersey"                   
 [7] "athlete_position_abbreviation"     "athlete_position_name"            
 [9] "athlete_short_name"                "blocks"                           
[11] "defensive_rebounds"                "did_not_play"                     
[13] "ejected"                           "field_goals_attempted"            
[15] "field_goals_made"                  "fouls"                            
[17] "free_throws_attempted"             "free_throws_made"                 
[19] "game_date"                         "game_date_time"                   
[21] "game_id"                           "home_away"                        
[23] "minutes"                           "offensive_rebounds"               
[25] "opponent_team_abbreviation"        "opponent_team_alternate_color"    

Wrote game roster files for season 2001-02 to data/game_rosters/2002


Fetching player box/roster for season 2002-03 (end-year 2003) ...

Wrote game roster files for season 2002-03 to data/game_rosters/2003


Fetching player box/roster for season 2003-04 (end-year 2004) ...

Wrote game roster files for season 2003-04 to data/game_rosters/2004


Fetching player box/roster for season 2004-05 (end-year 2005) ...

Wrote game roster files for season 2004-05 to data/game_rosters/2005


Fetching player box/roster for season 2005-06 (end-year 2006) ...

Wrote game roster files for season 2005-06 to data/game_rosters/2006


Fetching player box/roster for season 2006-07 (end-year 2007) ...

Wrote game roster files for season 2006-07 to data/game_rosters/2007


Fetching player box/roster for season 2007-08 (end-year 2008) ...

Wrote game roster files for season 2007-08 to data/game_rosters/2008


Fetching player box/roster for season 2008-09 (end-year 2009) ...

Wrote game roster files for season